## Transformer Optimization Experiment

This notebook performs an optimization experiment on the transformer
model used for sentiment classification.

First model configuration:
DistilBERT trained for **2 epochs**

Optimization experiment:
DistilBERT trained for **3 epochs**

The goal is to determine whether increasing the number of training
epochs improves classification performance.

In [1]:
#Import required libraries
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

#Device check
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

#Load processed data
train_df = pd.read_csv("../data/train.csv")
test_df  = pd.read_csv("../data/test.csv")

train_df.head()

CUDA available: True
GPU: NVIDIA GeForce GTX 1650


,Score,Text,sentiment,label
0,1,Eukanuba is a well-respected name in pet foods...,negative,0
1,1,We tried this with our 10-month-old son and he...,negative,0
2,3,I thought I was getting the standard package t...,neutral,1
3,3,"This food may be good for my cats, but two of ...",neutral,1
4,5,This is my favorite coffee outside of Italy. I...,positive,2


In [3]:
#Dataset Preparation
#Tokenization
#Load DistilBERT tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#Convert pandas DataFrame to HuggingFace dataset format
train_dataset = Dataset.from_pandas(train_df[["Text", "label"]])
test_dataset  = Dataset.from_pandas(test_df[["Text", "label"]])

def tokenize_function(examples):
    """
    Tokenize review text for transformer model input.

    Parameters
    examples : dict
        Dictionary containing review text.

    Returns
    dict
        Tokenized representation including attention masks.
    """
    
    return tokenizer(
        examples["Text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

#Apply tokenization to the dataset
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

#Remove raw text column - model uses tokens
train_dataset = train_dataset.remove_columns(["Text"])
test_dataset  = test_dataset.remove_columns(["Text"])

#Convert to PyTorch tensors
train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/102336 [00:00<?, ? examples/s]

Map:   0%|          | 0/25584 [00:00<?, ? examples/s]

In [5]:
#Model Setup

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

def compute_metrics(eval_pred) -> dict:
     """
    Compute evaluation metrics during training.

    Returns
    dict
        Accuracy, precision, recall and F1.
    """
    
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted"
    )
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

#Configure training parameters

training_args = TrainingArguments(
    output_dir="./results_epoch3",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3, #The number of epochs is increased from 2 → 3
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=200,
)

#Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

#Save model
trainer.save_model("../models/distilbert_3class_v2_optimized")
tokenizer.save_pretrained("../models/distilbert_3class_v2_optimized")
print("model saved")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.526877,0.502939,0.792097,0.794869,0.814761,0.792097
2,0.360559,0.458117,0.831731,0.831786,0.831865,0.831731
3,0.280846,0.499844,0.834271,0.834341,0.834470,0.834271


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model saved
